In [ ]:
import os
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt


#===================================
IMG_SIZE = 128
BATCH_SIZE = 8
EPOCHS = 10
#===================================
TRAIN_SAMPELS = 300
TETS_SAMPELS = 60

AUTOTUNE = tf.data.AUTOTUNE

#===================================
# run in google colab
# save in google drive

# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('oxford_iiit_pet.zip', '/content/drive/MyDrive/')
# print("فایل با موفقیت به گوگل درایو منتقل شد!")

# donload file and move to 
# C:\Users\mohse\tensorflow_datasets\oxford_iiit_pet\4.0.0


#===================================
# import dataset
#===================================

ds, info = tfds.load('oxford_iiit_pet', with_info=True)


#===================================
# preprocess
#===================================

def preprocess(sample):
    image = sample["image"]
    mask = sample["segmentation_mask"]

    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    mask = tf.image.resize(mask, (IMG_SIZE, IMG_SIZE), method="nearest")

    image = tf.cast(image, tf.float32) / 255.0

    mask = tf.where(mask == 1, 1.0, 0.0)
    mask = tf.cast(mask, tf.float32)

    return image, mask

#====================================
# pipline tran and test
#====================================

train_ds = (
    ds["train"]
    .map(preprocess, num_parallel_calls = AUTOTUNE)
    .shuffle(512)
    .take(TRAIN_SAMPELS)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    ds["test"]
    .map(preprocess, num_parallel_calls = AUTOTUNE)
    .take(TETS_SAMPELS)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)



In [10]:
#===================================
# model (Unet)
#===================================

def conv_block(x, filters):
    x= tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x= tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    return x


inputs = tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

#===================================
# Encoder
#===================================

c1 = conv_block(inputs, 16)
p1 = tf.keras.layers.MaxPooling2D()(c1)

c2 = conv_block(p1, 32)
p2 = tf.keras.layers.MaxPooling2D()(c2)

b = conv_block(p2, 64)


#===================================
# Decoder
#===================================

u2 = tf.keras.layers.UpSampling2D()(b)
u2 = tf.keras.layers.Concatenate()([u2, c2])
c3 = conv_block(u2, 32)

u1 = tf.keras.layers.UpSampling2D()(c3)
u1 = tf.keras.layers.Concatenate()([u1, c1])
c4 = conv_block(u1, 16)


output =  tf.keras.layers.Conv2D(1,1, activation="sigmoid")(c4)



model = tf.keras.Model(inputs, output)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 128, 128,  │        448 │ input_layer_2[0]… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_13 (Conv2D)  │ (None, 128, 128,  │      2,320 │ conv2d_12[0][0]   │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 64, 64,    │          0 │ conv2d_13[0][0]   │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_14 (Conv2D)  │ (None, 64, 64,    │      4,640 │ max_pooling2d_2[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_15 (Conv2D)  │ (None, 64, 64,    │      9,248 │ conv2d_14[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 32, 32,    │          0 │ conv2d_15[0][0]   │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_16 (Conv2D)  │ (None, 32, 32,    │     18,496 │ max_pooling2d_3[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 32, 32,    │     36,928 │ conv2d_16[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_2     │ (None, 64, 64,    │          0 │ conv2d_17[0][0]   │
│ (UpSampling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 64, 64,    │          0 │ up_sampling2d_2[… │
│ (Concatenate)       │ 96)               │            │ conv2d_15[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 64, 64,    │     27,680 │ concatenate_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 64, 64,    │      9,248 │ conv2d_18[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_3     │ (None, 128, 128,  │          0 │ conv2d_19[0][0]   │
│ (UpSampling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 128, 128,  │          0 │ up_sampling2d_3[… │
│ (Concatenate)       │ 48)               │            │ conv2d_13[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 128, 128,  │      6,928 │ concatenate_3[0]… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 128, 128,  │      2,320 │ conv2d_20[0][0] 

 Total params: 118,273 (462.00 KB)

 Trainable params: 118,273 (462.00 KB)

 Non-trainable params: 0 (0.00 B)